In [4]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ================= 1. 数据加载与预处理 =================
file_path = '智能家居环境控制系统数据集.xlsx'
df = pd.read_excel(file_path)

# 去除所有列名前后的隐藏空格
df.columns = df.columns.str.strip()

# 【关键】将时间戳转换为 datetime 格式，并提取小时数用于分箱
df['时间戳'] = pd.to_datetime(df['时间戳'])
df['hour'] = df['时间戳'].dt.hour

print("✅ 成功加载数据集，开始分析...")
print("=" * 60)

# ================= 2. 一、用户环境偏好分析 =================
print("\n【一、用户环境偏好分析】")

# 定义时段分箱规则 (对应Excel公式: 0-6, 6-12, 12-18, 18-24)
bins = [0, 6, 12, 18, 24]
labels = ['0-6点', '6-12点', '12-18点', '18-24点']
df['time_period'] = pd.cut(df['hour'], bins=bins, labels=labels, right=False)

# 计算各时段的平均温度、湿度和光照水平
env_pref = df.groupby('time_period')[['温度', '湿度', '光照水平']].mean().round(2)

for period, row in env_pref.iterrows():
    print(f"• {period}: 平均温度 {row['温度']}℃, 平均湿度 {row['湿度']}%, 平均光照 {row['光照水平']}")

print("\n💡 偏好总结:")
print("   - 温度偏好: 晚上(18-24点)的平均温度最高，早上(0-6点)最低，符合夜间保暖/清晨凉爽的习惯。")
print("   - 湿度偏好: 整体波动不大。")
print("   - 光照偏好: 早上光照水平最高，晚上最低，说明用户在白天偏好强光，夜晚偏好柔和光线。")

# ================= 3. 二、系统响应时间分析 =================
print("\n【二、系统响应时间分析】")

avg_resp_time = df['响应时间'].mean()
print(f"• 平均响应时间: 约 {avg_resp_time:.2f} 秒")

# 【核心逻辑】使用 Pandas 计算环境因素与响应时间的相关系数
corr_temp = df['温度'].corr(df['响应时间'])
corr_hum = df['湿度'].corr(df['响应时间'])
corr_light = df['光照水平'].corr(df['响应时间'])

print("\n📊 影响因素相关性分析 (Pearson相关系数):")
print(f"   1. 环境温度 vs 响应时间: {corr_temp:.4f} (基本无影响)")
print(f"   2. 湿度波动 vs 响应时间: {corr_hum:.4f} (影响不明显)")
print(f"   3. 光照水平 vs 响应时间: {corr_light:.4f} (光照越高响应略快，但影响非常有限)")
print("\n⚠️ 其他可能影响因素: 网络连接状态、传感器灵敏度、应用程序固件版本等。")

# ================= 4. 三、能源消耗分析 =================
print("\n【三、能源消耗分析】")

avg_energy = df['能源消耗'].mean()
print(f"• 平均能耗: 约 {avg_energy:.2f} 单位")

# 计算环境因素与能耗的相关系数
e_corr_temp = df['温度'].corr(df['能源消耗'])
e_corr_hum = df['湿度'].corr(df['能源消耗'])
e_corr_light = df['光照水平'].corr(df['能源消耗'])

print("\n🔍 节能潜力识别:")
print(f"   1. 温度变化对能耗影响: 相关系数 {e_corr_temp:.4f} (无明显影响，无需针对温度调节做节能优化)")
print(f"   2. 湿度变化对能耗影响: 相关系数 {e_corr_hum:.4f} (影响极小，暂无节能潜力)")
print(f"   3. 光照强度对能耗影响: 相关系数 {e_corr_light:.4f} (光照强时能耗略有增加)")
print("\n💡 节能建议: 可在光照强时段适当降低设备功率或减少照明补光，以降低整体能耗。")



✅ 成功加载数据集，开始分析...

【一、用户环境偏好分析】
• 0-6点: 平均温度 24.76℃, 平均湿度 49.88%, 平均光照 554.45
• 6-12点: 平均温度 24.8℃, 平均湿度 49.56%, 平均光照 544.89
• 12-18点: 平均温度 24.91℃, 平均湿度 50.07%, 平均光照 545.72
• 18-24点: 平均温度 25.16℃, 平均湿度 49.69%, 平均光照 538.78

💡 偏好总结:
   - 温度偏好: 晚上(18-24点)的平均温度最高，早上(0-6点)最低，符合夜间保暖/清晨凉爽的习惯。
   - 湿度偏好: 整体波动不大。
   - 光照偏好: 早上光照水平最高，晚上最低，说明用户在白天偏好强光，夜晚偏好柔和光线。

【二、系统响应时间分析】
• 平均响应时间: 约 3.02 秒

📊 影响因素相关性分析 (Pearson相关系数):
   1. 环境温度 vs 响应时间: 0.0036 (基本无影响)
   2. 湿度波动 vs 响应时间: 0.0160 (影响不明显)
   3. 光照水平 vs 响应时间: -0.0324 (光照越高响应略快，但影响非常有限)

⚠️ 其他可能影响因素: 网络连接状态、传感器灵敏度、应用程序固件版本等。

【三、能源消耗分析】
• 平均能耗: 约 1.02 单位

🔍 节能潜力识别:
   1. 温度变化对能耗影响: 相关系数 -0.0137 (无明显影响，无需针对温度调节做节能优化)
   2. 湿度变化对能耗影响: 相关系数 -0.0304 (影响极小，暂无节能潜力)
   3. 光照强度对能耗影响: 相关系数 0.0626 (光照强时能耗略有增加)

💡 节能建议: 可在光照强时段适当降低设备功率或减少照明补光，以降低整体能耗。
